# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.io as pio
pio.renderers.default = 'colab+plotly_mimetype'   # shows in Colab AND saves a VS Code-readable copy

# Dataset: Global Energy Mix by Country and Source
try:
    df = pd.read_csv('../data/global_energy_mix.csv')   # repo structure
except FileNotFoundError:
    df = pd.read_csv('global_energy_mix.csv')           # Colab

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1

df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Story: which region burns the most fossil energy?
region_twh = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
top_region = region_twh.index[0]
region_share = region_twh.iloc[0] / region_twh.sum()

# CVD-safe course palette: orange / blue / yellow (different hue AND lightness)
fossil_colors = {'Coal': '#E07B39', 'Oil': '#2E75B6', 'Natural Gas': '#F5C842'}

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],   # order = hierarchy; parents are summed automatically
    values='TWh',                           # additive -> area is meaningful at every level
    color='Source',
    color_discrete_map=fossil_colors,
    labels={'TWh': 'Energy (TWh)'},
    height=700,
    template='plotly_white')

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',   # label first, then value - TWh only, no %
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white')

# Grey out parent nodes (Region, Country) - only the source leaves keep colour
fig.data[0].marker.colors = [
    c if c in fossil_colors.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors]

fig.update_layout(
    # Source names coloured like the chart - the title works as the legend
    title=(f"{top_region} is the most fossil-dependent region, burning {region_share:.0%} of the world's fossil energy — "
           "<span style='color:#E07B39'><b>coal</b></span> leads, "
           "<span style='color:#2E75B6'><b>oil</b></span> and "
           "<span style='color:#F5C842'><b>natural gas</b></span> trail"),
    font=dict(family='Arial', size=12),
    margin=dict(t=60, l=10, r=10, b=10))
fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 - charts

low = (df.loc[df['Source_Type'] == 'Low-carbon']
         .groupby('Country', as_index=False)['TWh'].sum())
low['All'] = 'Low-carbon'          # dummy root node
low = low.sort_values('TWh')       # ascending: px draws the first row at the bottom -> largest on top

# --- Treemap ---
fig = px.treemap(
    low,
    path=['All', 'Country'],
    values='TWh',
    title="<span style='color:#2E75B6'><b>Low-carbon</b></span> energy (nuclear + hydro) by country, TWh",
    height=550,
    template='plotly_white'
)
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',   # label first, then value
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white'
)
# One blue for all countries, grey root
fig.data[0].marker.colors = ['#DDDDDD' if lbl == 'Low-carbon' else '#2E75B6'
                             for lbl in fig.data[0].labels]
fig.update_layout(font=dict(family='Arial', size=12), margin=dict(t=60, l=10, r=10, b=10))
fig.show()

# --- Horizontal bar ---
leader, second = low.iloc[-1], low.iloc[-2]

fig = px.bar(
    low,
    x='TWh', y='Country',
    orientation='h',                       # label read first, then the bar
    text='TWh',
    color_discrete_sequence=['#2E75B6'],   # CVD-safe blue, same as the treemap
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    height=550,
    template='plotly_white',
    title=f"{leader['Country']} leads <span style='color:#2E75B6'><b>low-carbon</b></span> energy with "
          f"{leader['TWh']:,.0f} TWh — {leader['TWh'] / second['TWh']:.1f}x {second['Country']} in second place"
)
fig.update_traces(texttemplate='%{x:,.0f}', textposition='outside')   # value at the end of each bar
fig.update_layout(
    font=dict(family='Arial', size=12),
    xaxis=dict(visible=False, range=[0, low['TWh'].max() * 1.12]),   # zero baseline; labels replace the axis
    yaxis=dict(title='')
)
fig.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The answer is: horizontal bar chart

Bars share a common zero baseline and are sorted, so the eye compares end points and reads the ranking top to bottom instantly, where France first, then Norway and Canada. In the treemap the same values are encoded as rectangle **areas** with different shapes (wide vs tall), and humans judge area far less accurately than length. Without the printed numbers, it is hard to tell whether Norway or Canada is second. The treemap shows "who is big" at a glance, but for ranking, the bar wins.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
# Task 3

# Region x Source TWh; pivot + melt keeps a 0 row for regions with no coal (Middle East)
fos = (df_fossil.pivot_table(index='Region', columns='Source', values='TWh', aggfunc='sum', fill_value=0)
                .reset_index()
                .melt(id_vars='Region', var_name='Source', value_name='TWh'))
fos['Share'] = fos['TWh'] / fos.groupby('Region')['TWh'].transform('sum')   # 100% stacked -> shares

# Sort regions by coal share, highest at the top
coal = fos.loc[fos['Source'] == 'Coal'].sort_values('Share', ascending=False)
region_order = coal['Region'].tolist()
top = coal.iloc[0]

# Labels on coal segments only (skip 0% for the Middle East)
fos['Label'] = np.where((fos['Source'] == 'Coal') & (fos['Share'] > 0),
                        fos['Share'].map('{:.0%}'.format), '')

fig = px.bar(
    fos,
    x='Share', y='Region',
    color='Source',
    text='Label',
    orientation='h',                                # label read first, then the bar
    color_discrete_map={'Coal': '#E07B39',          # same orange as Task 1 = the story
                        'Oil': '#BBBBBB',           # greys = context
                        'Natural Gas': '#DDDDDD'},
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   # coal on the baseline
    labels={'Share': 'Share of fossil energy', 'Source': ''},
    hover_data={'Label': False, 'Share': ':.0%', 'TWh': ':,.0f'},
    height=450,
    template='plotly_white',
    title=f"{top['Region']} relies most on <span style='color:#E07B39'><b>coal</b></span> — "
          f"{top['Share']:.0%} of its fossil energy"
)

fig.update_traces(
    textposition='inside', insidetextanchor='start',   # label at the left edge of the segment
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1     # thin white separators
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.12),          # needed to tell the two greys apart
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title='')
)

# Every bar sums to 100% - the hidden x-axis no longer says so
fig.add_annotation(x=1, y=0, xref='x', yref='paper', xanchor='right', yanchor='top', yshift=-4,
                   text='100%', showarrow=False, font=dict(size=11, color='grey'))
fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

**Treemap only:** *How big is each country's fossil use in absolute terms, and where does it sit in the hierarchy?* Here, China's coal alone (6,929 TWh) is more than Brazil's entire fossil use (4,187 TWh). The stacked bar converts everything to 100%, so absolute size disappears.

**Stacked bar only:** *Which region depends most on coal, and by how much more than the others?* Here, Africa 78% vs Asia 49% vs Europe 6%. All coal segments start at the same baseline, so the shares can be ranked precisely by length. In the treemap, coal rectangles are scattered across regions in different shapes, so their shares can't be compared.

*Caveat:* in this dataset Africa = South Africa only and the Middle East = Saudi Arabia only, so these regional shares reflect one country each.